# 高阶Matmul API

## 概述

上一节我们用`asc.data_copy`/`asc.load_data`/`asc.mmad`/`asc.fixpipe`四个基础API实现了带K方向分块累加的矩阵乘法，感受到了手动管理存储层级和数据搬运的繁琐。本节引入pyasc的高阶Matmul API——`asc.adv.Matmul`体系，它封装了数据搬运、分块迭代、多核并行等全部细节，让Matmul开发从"指令级"提升到"框架级"。

### 学习目标

1. 理解高阶API体系：`MatmulType`、`Matmul`、`register_matmul`、`TCubeTiling`
2. 掌握`host.MultiCoreMatmulTiling`生成Tiling参数的方法
3. 掌握高阶API的7步调用流程
4. 通过对比基础API实现，理解高阶API的封装价值

In [ ]:
# 环境初始化
!mkdir -p Sources/04.03

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

---
# 1. 从基础到高阶

回顾上一节的基础实现，完成一次矩阵乘法需要：

| 基础步骤 | 代码 | 痛点 |
| --- | --- | --- |
| 分配L1/L0A/L0B/CO1 | `asc.LocalTensor(pos=A1/B1/A2/B2/CO1, ...)` | 需手动计算tile_size |
| 搬运A/B到L1（ND→NZ） | `asc.data_copy(a_l1, a_gm, intri_params=nd2nz_params)` | 需手动设置Nd2NzParams的8个参数 |
| 流水同步 MTE2→MTE1 | `asc.set_flag(MTE2_MTE1, ...)` / `asc.wait_flag(...)` | 需手动获取event_id并插入同步 |
| 搬运A/B到L0A/L0B | `asc.load_data(a_l0a, a_l1, load_params)` | 需手动设置LoadData2DParams |
| 流水同步 MTE1→M | `asc.set_flag(MTE1_M, ...)` / `asc.wait_flag(...)` | 需手动插入同步 |
| 执行mmad | `asc.mmad(c_l0c, a_l0a, b_l0b, mmad_params)` | 需手动设置MmadParams，含cmatrix_init_val累加控制 |
| 流水同步 M→FIX | `asc.set_flag(M_FIX, ...)` / `asc.wait_flag(...)` | 需手动插入同步 |
| 搬运结果到GM（NZ→ND） | `asc.fixpipe(c_gm, c_l0c, fixpipe_params)` | 需手动设置FixpipeParamsV220 |
| 多核并行 | ❌ 不支持 | 基础API不含多核切分 |
| K轴分块迭代 | 手动for循环 + K块偏移 + MTE1→MTE2迭代间同步 | 需手动编写循环、计算偏移、插入迭代间同步 |
| 尾块处理 | ❌ 不支持 | 需手动判断边界 |

**高阶API的价值**：上述11个痛点全部由`asc.adv.Matmul`自动处理（包括流水同步），开发者只需声明输入/输出类型和Tiling参数。

高阶API由Host侧的Tiling生成和Kernel侧的Matmul执行两部分协同组成，其整体架构如下图所示：

<img src="./images/04.03_high_level_matmul_api/high_level_api_arch.png" alt="高阶Matmul API组件架构" width="650px">

---
# 2. 核心API体系

高阶Matmul API由4个核心组件构成：

### 2.1 MatmulType — 类型描述

描述矩阵的类型信息，包括存储位置、数据格式、数据类型和是否转置。

```python
asc.adv.MatmulType(
    position,    # TPosition.GM
    format,      # CubeFormat.ND
    dtype,       # asc.float16 / asc.float32
    is_trans     # 是否转置（可选，默认False）
)
```

### 2.2 Matmul — 计算对象

创建Matmul计算对象，通过MatmulType声明A/B/C（可选Bias）的类型。

```python
matmul = asc.adv.Matmul(
    a=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, a_dtype, False),
    b=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, b_dtype, False),
    c=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, c_dtype),
    bias=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, bias_dtype),  # 可选
)
```

### 2.3 register_matmul — 初始化

将Matmul对象与TPipe、workspace和Tiling关联，完成内部资源分配。

```python
asc.adv.register_matmul(pipe, workspace, matmul, tiling)
```

### 2.4 TCubeTiling — 切分参数

由Host侧Tiling API生成的结构体，包含多核切分和分块计算的全部参数。

| 关键字段 | 说明 |
| --- | --- |
| `used_core_num` | 实际使用的核数 |
| `m` / `n` / `k_a` / `k_b` | 矩阵M/N/K维度 |
| `single_core_m` / `single_core_n` | 单核负责的M/N维度大小 |
| `base_m` / `base_n` | 单次mmad计算的baseM/baseN |
| `is_bias` | 是否启用Bias |
| `m.ceildiv(x)` | 向上取整除法 |

---
# 3. Tiling详解

### 3.1 为什么需要Tiling？

Cube计算单元的本地存储（L1/L0A/L0B/L0C）容量有限，无法一次性放入整个矩阵。必须将大矩阵切成小块（Tile），每次搬运一个Tile到本地存储计算，最终拼成完整结果。Tiling就是**决定如何切分**的过程。

```
GM（大仓库，GB级）
  ↓ 按 single_core_m × single_core_n 切分给每个核
单核分片（中转站）
  ↓ 按 base_m × base_n 切分给每次mmad
L0（工作台，几十KB）
  ↓ mmad计算
结果
```

### 3.2 两级Tiling参数

pyasc的Tiling分为**多核切分**和**单核内迭代**两级：

| 层级 | 参数 | 含义 | 类比 |
| --- | --- | --- | --- |
| **多核切分** | `single_core_m` / `single_core_n` | 每个核负责的M/N维度大小 | 每个工人分到的货物量 |
| **多核切分** | `used_core_num` | 实际使用的核数 | 工人总数 |
| **单核迭代** | `base_m` / `base_n` | 单次mmad计算的M/N大小（分形块的整数倍） | 工人一次处理的货物量 |
| **K轴迭代** | `k_a` / `k_b` | A/B矩阵的K维度大小 | 货物的深度 |
| **K轴迭代** | `single_core_k` | 单核K轴迭代总长 | 每个工人负责的深度 |
| **K轴迭代** | `base_k` | 单次mmad的K大小 | 每次搬运的深度 |

### 3.3 多核切分示意

以 M=256, N=256, `single_core_m=64`, `single_core_n=64` 为例，矩阵C被切分为 4×4=16 个分片，分配给16个核：

```
         N=256
    ┌────┬────┬────┬────┐
    │核0 │核4 │核8 │核12│  ← single_core_n=64
    ├────┼────┼────┼────┤
M=  │核1 │核5 │核9 │核13│
256 ├────┼────┼────┼────┤
    │核2 │核6 │核10│核14│
    ├────┼────┼────┼────┤
    │核3 │核7 │核11│核15│  ← single_core_m=64
    └────┴────┴────┴────┘

m_single_blocks = ceildiv(256, 64) = 4
核5: m_index=5%4=1, n_index=5//4=1
```

### 3.4 K轴迭代示意

每个核内部，K维度被切分为多个`base_k`大小的块，逐块搬运并累加到L0C：

```
A[M, K] 沿K轴切分：
    ┌──────┬──────┬──────┬───┐
    │ K块0 │ K块1 │ K块2 │...│  ← 每块 base_k 列
    └──────┴──────┴──────┴───┘

B[K, N] 沿K轴切分：
    ┌──────┐
    │ K块0 │  ← 每块 base_k 行
    ├──────┤
    │ K块1 │
    ├──────┤
    │ K块2 │
    ├──────┤
    │ ...  │
    └──────┘

C = A_0×B_0 + A_1×B_1 + A_2×B_2 + ...（累加到L0C）
```

高阶API的`iterate_all`自动完成上述K轴迭代和累加，开发者无需手动编写循环。

### 3.5 尾块处理

当矩阵维度不能被`single_core_m`或`single_core_n`整除时，边缘的分片是不满的"尾块"。例如 M=1000, `single_core_m=64`：

```
1000 / 64 = 15 块（满块）+ 余量 40（尾块）
tail_m = 1000 - 15 * 64 = 40
```

通过`set_tail(tail_m, tail_n)`告知高阶API当前核的尾块大小，API内部自动处理边界，避免越界访问。

### 3.6 Tiling生成代码

Tiling参数由Host侧的`host.MultiCoreMatmulTiling`生成，它根据矩阵形状、数据类型和核数自动计算最优切分策略。

```python
matmul_tiling = host.MultiCoreMatmulTiling(host.get_ascendc_platform())

# 步骤1：声明A/B/C/Bias的类型
matmul_tiling.set_a_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT16, False)
matmul_tiling.set_b_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT16, False)
matmul_tiling.set_c_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT)
matmul_tiling.set_bias_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT)  # 声明Bias类型，即使当前未启用

# 步骤2：设置矩阵形状和核数
matmul_tiling.set_dim(USE_CORE_NUM)        # 核数
matmul_tiling.set_org_shape(m, n, k)       # 原始形状
matmul_tiling.set_shape(m, n, k)           # 计算形状

# 步骤3：配置Bias和缓冲区
matmul_tiling.enable_bias(False)           # 是否启用Bias
matmul_tiling.set_buffer_space(-1, -1, -1) # 自动分配缓冲区

# 步骤4：生成Tiling
tiling = asc.adv.TCubeTiling()
matmul_tiling.get_tiling(tiling)
```

> **注意**：
> - Host侧的TPosition/CubeFormat/DataType使用`host.`前缀（如`host.TPosition.GM`），与kernel侧的`asc.TPosition.GM`不同。
> - `set_bias_type`用于声明Bias的数据类型。即使当前`enable_bias(False)`，也建议预先声明，后续启用Bias时无需额外配置（04.04节将启用Bias）。

---
# 4. 基本调用流程

高阶API的完整调用流程分为7步：

```
步骤1: 创建GlobalTensor并绑定GM地址
步骤2: 创建TPipe对象
步骤3: 创建Matmul对象（声明A/B/C/Bias类型）
步骤4: register_matmul初始化（关联pipe/workspace/tiling）
步骤5: set_tensor_a / set_tensor_b（设置输入矩阵）
步骤6: set_tail（设置尾块，可选）→ iterate_all（执行计算）
步骤7: end（释放资源）
```

下图直观展示了这7个步骤的执行顺序和数据流向：

<img src="./images/04.03_high_level_matmul_api/call_flow.png" alt="高阶API 7步调用流程" width="350px">

### 多核偏移计算

在多核并行场景下，每个核需要计算自己负责的数据分片偏移：

```python
block_idx = asc.get_block_idx()
m_single_blocks = tiling.m.ceildiv(tiling.single_core_m)  # M方向分块数
m_index = block_idx % m_single_blocks                      # M方向索引
n_index = block_idx // m_single_blocks                     # N方向索引

offset_a = m_index * tiling.k_a * tiling.single_core_m
offset_b = n_index * tiling.single_core_n
offset_c = m_index * tiling.n * tiling.single_core_m + n_index * tiling.single_core_n
```

---
# 5. 完整代码实现

使用高阶API实现256×256的多核并行Matmul，对比上一节基础API的繁琐实现。

**计算规格**：A[256,256] fp16 × B[256,256] fp16 → C[256,256] fp32，8核并行。

In [ ]:
%%writefile Sources/04.03/matmul_high_level_api.py
# Copyright (c) 2025 Huawei Technologies Co., Ltd.
# CANN Open Software License Agreement Version 2.0

from typing import Tuple
import logging
import argparse
import torch

try:
    import torch_npu
except ModuleNotFoundError:
    pass

import asc
import asc.runtime.config as config
import asc.lib.runtime as rt
import asc.lib.host as host

logging.basicConfig(level=logging.INFO)

USE_CORE_NUM = 8
M_DIM = 256
N_DIM = 256
K_DIM = 256


@asc.jit(always_compile=True)
def matmul_high_level_kernel(a: asc.GlobalAddress, b: asc.GlobalAddress, c: asc.GlobalAddress,
                              tiling: asc.adv.TCubeTiling, workspace: asc.GlobalAddress):
    # 多核偏移计算：每个核处理不同的数据分片
    offset_a, offset_b, offset_c, tail_m, tail_n = calc_offsets(tiling)

    # 步骤1：创建GlobalTensor并绑定GM地址（带偏移）
    a_global = asc.GlobalTensor()
    b_global = asc.GlobalTensor()
    c_global = asc.GlobalTensor()
    a_global.set_global_buffer(a + offset_a)
    b_global.set_global_buffer(b + offset_b)
    c_global.set_global_buffer(c + offset_c)

    # 步骤2：创建TPipe
    pipe = asc.TPipe()

    # 步骤3：创建Matmul对象（声明A/B/C类型）
    matmul = asc.adv.Matmul(
        a=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, a_global.dtype, False),
        b=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, b_global.dtype, False),
        c=asc.adv.MatmulType(asc.TPosition.GM, asc.CubeFormat.ND, c_global.dtype),
    )

    # 步骤4：register_matmul初始化
    asc.adv.register_matmul(pipe, workspace, matmul, tiling)

    # 步骤5-7：设置输入、执行计算、结束
    if asc.get_block_idx() < tiling.used_core_num:
        matmul.set_tensor_a(a_global, False)
        matmul.set_tensor_b(b_global, False)
        matmul.set_tail(tail_m, tail_n)
        matmul.iterate_all(c_global)
        matmul.end()

    asc.pipe_barrier(asc.PipeID.PIPE_ALL)


@asc.jit
def calc_offsets(tiling: asc.adv.TCubeTiling) -> Tuple[int, int, int, int, int]:
    block_idx = asc.get_block_idx()
    m_single_blocks = tiling.m.ceildiv(tiling.single_core_m)
    m_index = block_idx % m_single_blocks
    n_index = block_idx // m_single_blocks
    offset_a = m_index * tiling.k_a * tiling.single_core_m
    offset_b = n_index * tiling.single_core_n
    offset_c = m_index * tiling.n * tiling.single_core_m + n_index * tiling.single_core_n
    # 尾块处理
    tail_m = tiling.m - m_index * tiling.single_core_m
    if tail_m >= tiling.single_core_m:
        tail_m = tiling.single_core_m
    tail_n = tiling.n - n_index * tiling.single_core_n
    if tail_n >= tiling.single_core_n:
        tail_n = tiling.single_core_n
    return offset_a, offset_b, offset_c, tail_m, tail_n


def generate_tiling(m, n, k) -> asc.adv.TCubeTiling:
    matmul_tiling = host.MultiCoreMatmulTiling(host.get_ascendc_platform())
    matmul_tiling.set_a_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT16, False)
    matmul_tiling.set_b_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT16, False)
    matmul_tiling.set_c_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT)
    matmul_tiling.set_bias_type(host.TPosition.GM, host.CubeFormat.ND, host.DataType.DT_FLOAT)
    matmul_tiling.set_dim(USE_CORE_NUM)
    matmul_tiling.set_org_shape(m, n, k)
    matmul_tiling.set_shape(m, n, k)
    matmul_tiling.enable_bias(False)
    matmul_tiling.set_buffer_space(-1, -1, -1)

    tiling = asc.adv.TCubeTiling()
    matmul_tiling.get_tiling(tiling)
    return tiling


def matmul_high_level_custom(backend: config.Backend, platform: config.Platform):
    config.set_platform(backend, platform)
    device = "npu" if config.Backend(backend) == config.Backend.NPU else "cpu"

    a = torch.randint(-5, 5, (M_DIM, K_DIM), device=device).to(torch.float16)
    b = torch.randint(-5, 5, (K_DIM, N_DIM), device=device).to(torch.float16)
    c = torch.zeros((M_DIM, N_DIM), dtype=torch.float32, device=device)

    tiling = generate_tiling(M_DIM, N_DIM, K_DIM)
    workspace = torch.zeros(16 * 1024 * 1024, dtype=torch.uint8, device=device)
    # MIX模式：核数为USE_CORE_NUM//2（AIC+AIV成对）
    matmul_high_level_kernel[USE_CORE_NUM // 2, rt.current_stream()](a, b, c, tiling, workspace)

    golden = torch.matmul(a.to(torch.float32), b.to(torch.float32))
    assert torch.allclose(c, golden, rtol=1e-3, atol=1e-3)
    logging.info(f"[INFO] 高阶API验证通过! M={M_DIM}, N={N_DIM}, K={K_DIM}, 核数={USE_CORE_NUM}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("-r", type=str, default="NPU", help="backend to run")
    parser.add_argument("-v", type=str, default=None, help="platform to run")
    args = parser.parse_args()
    backend = args.r
    platform = args.v
    if backend not in config.Backend.__members__:
        raise ValueError(f"Unsupported Backend! Supported: {list(config.Backend.__members__.keys())}")
    backend = config.Backend(backend)
    if platform is not None:
        platform_values = [p.value for p in config.Platform]
        if platform not in platform_values:
            raise ValueError(f"Unsupported Platform! Supported: {platform_values}")
        platform = config.Platform(platform)
    logging.info("[INFO] start process sample matmul_high_level_api.")
    matmul_high_level_custom(backend, platform)
    logging.info("[INFO] Sample matmul_high_level_api run success.")

In [ ]:
# 运行高阶API示例（NPU模式）
!python3 Sources/04.03/matmul_high_level_api.py -r NPU

---
# 6. 基础vs高阶对比

| 维度 | 基础API（04.02） | 高阶API（04.03） |
| --- | --- | --- |
| **代码行数** | ~90行核函数 | ~30行核函数 |
| **存储管理** | 手动分配L1/L0A/L0B/CO1 | 自动管理 |
| **数据搬运** | 手动data_copy + load_data + fixpipe | 自动搬运 |
| **格式转换** | 手动处理ND→NZ / NZ→ND | 自动转换 |
| **流水同步** | 手动set_flag/wait_flag（4组，含迭代间同步） | 自动同步 |
| **多核并行** | ❌ 不支持 | ✅ 自动切分 |
| **K轴分块** | 手动for循环 + cmatrix_init_val累加控制 | ✅ 自动迭代 |
| **尾块处理** | ❌ 不支持 | ✅ set_tail |
| **Bias支持** | ❌ 需手动实现 | ✅ set_bias |
| **参数配置** | Nd2NzParams/LoadData2DParams/MmadParams/FixpipeParamsV220 | TCubeTiling（自动生成） |

> 高阶API将开发者从硬件细节中解放出来，专注于算子逻辑和性能调优。后续小节将在此基础上学习Cube Only和MIX两种执行模式。

---
# 7. 小结

本节介绍了pyasc高阶Matmul API体系，要点回顾：

| 组件 | 作用 | 调用方式 |
| --- | --- | --- |
| `MatmulType` | 描述矩阵类型 | `asc.adv.MatmulType(pos, format, dtype, is_trans)` |
| `Matmul` | 创建计算对象 | `asc.adv.Matmul(a=, b=, c=, bias=)` |
| `register_matmul` | 初始化 | `asc.adv.register_matmul(pipe, workspace, matmul, tiling)` |
| `TCubeTiling` | 切分参数 | `host.MultiCoreMatmulTiling` → `get_tiling()` |
| `set_tensor_a/b` | 设置输入 | `matmul.set_tensor_a(a_global, is_trans)` |
| `set_tail` | 尾块处理 | `matmul.set_tail(tail_m, tail_n)` |
| `iterate_all` | 执行计算 | `matmul.iterate_all(c_global)` |
| `end` | 释放资源 | `matmul.end()` |

---

## 课后练习

### 选择题

**1.** `asc.adv.register_matmul` 的核心作用是？

- A. 创建Matmul对象
- B. 初始化Matmul对象，关联TPipe/workspace/Tiling
- C. 执行矩阵乘计算
- D. 生成Tiling参数

**2.** `host.MultiCoreMatmulTiling.set_dim(N)` 的参数N表示什么？

- A. 矩阵的N维度大小
- B. 多核并行的核数
- C. 单核计算的baseN
- D. workspace大小

### 填空题

**3.** 执行单核全部计算并输出结果的高阶API方法是 ______ 。

**4.** 高阶API的7步调用流程是：创建GlobalTensor → 创建TPipe → ______ → register_matmul → set_tensor_a/b → set_tail + iterate_all → end。

**5.** Tiling中 `single_core_m` 表示 ______ ，`base_m` 表示 ______ 。当矩阵M=1000、`single_core_m=64`时，M方向共有 ______ 个满块和 ______ 个尾块（尾块大小为 ______ ）。

---

> 点击下方查看答案

In [ ]:
!cat ./answer/04.03_high_level_matmul_api/practice_answers.md